In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np

## 0. Architecture
```
Input → Embedding + PosEnc → [Attn + FFN] × N → LayerNorm → [CLS] → Classifier
```

## 1. Scaled Dot-Product Attention
$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q, K, V: (batch, heads, seq_len, d_k)
    mask:    (batch, 1, seq_len, seq_len)  -- attention_mask mở rộng
    Returns: (batch, heads, seq_len, d_k)
    """
    d_k = Q.shape[-1]
    
    # TODO 1: Tính Q·Kᵀ  (matrix multiply)
    scores =   np.matmul(Q, K.T)
    
    # TODO 2: Scale với √d_k
    scores = scores / np.sqrt(d_k)  # <- thay dòng này
    
    # TODO 3: Nếu có mask, set vị trí mask=0 thành -inf
    # Gợi ý: scores = scores.masked_fill(mask == 0, float('-inf'))
    if mask is not None:
        pass  # <- thay dòng này
    
    # TODO 4: Softmax trên chiều cuối cùng (dim=-1)
    attn_weights = None  # <- thay dòng này
    
    # TODO 5: Nhân với V
    output = None  # <- thay dòng này
    
    return output


# === Test sau khi implement ===
# Q = K = V = torch.randn(2, 1, 4, 8)  # batch=2, heads=1, seq=4, d_k=8
# mask = torch.ones(2, 1, 4, 4)
# mask[:, :, 2:, 2:] = 0  # block 2 token cuối
# out = scaled_dot_product_attention(Q, K, V, mask)
# print(out.shape)  # Expected: (2, 1, 4, 8)
# print(out)

## 2. Multi-Head Attention
Chạy H attention song song, concat kết quả rồi project qua W_O.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        """
        d_model:   kích thước embedding (vd: 256)
        num_heads: số đầu attention (vd: 8)
        d_k:       d_model / num_heads (vd: 32)
        """
        super().__init__()
        assert d_model % num_heads == 0, "d_model phải chia hết cho num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # TODO: Khởi tạo 4 Linear layers cho Q, K, V, và Output
        # Gợi ý: nn.Linear(d_model, d_model)
        self.W_Q = None  # <- thay
        self.W_K = None  # <- thay
        self.W_V = None  # <- thay
        self.W_O = None  # <- thay
        
    def forward(self, x, mask=None):
        """
        x:    (batch, seq_len, d_model)
        mask: (batch, 1, 1, seq_len) -- attention_mask, cần mở rộng
        Returns: (batch, seq_len, d_model)
        """
        batch_size, seq_len, _ = x.shape
        
        # TODO 1: Project Q, K, V từ x
        Q = None  # <- (batch, seq_len, d_model)
        K = None  # <- (batch, seq_len, d_model)
        V = None  # <- (batch, seq_len, d_model)
        
        # TODO 2: Reshape thành (batch, num_heads, seq_len, d_k)
        # Gợi ý: dùng .view() hoặc .reshape()
        Q = None  # <- thay
        K = None  # <- thay
        V = None  # <- thay
        
        # TODO 3: Gọi scaled_dot_product_attention
        attn_out = None  # <- thay
        
        # TODO 4: Reshape về (batch, seq_len, d_model)
        # Gợi ý: .transpose(1,2) rồi .reshape(batch, seq_len, d_model)
        attn_out = None  # <- thay
        
        # TODO 5: Project output với W_O
        output = None  # <- thay
        
        return output

## 3. Feed-Forward Network
$$\text{FFN}(x) = \text{GELU}(xW_1 + b_1)W_2 + b_2 \qquad d_{ff} = 4 \times d_{model}$$

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        # TODO: Linear d_model -> d_ff -> GELU -> Linear d_ff -> d_model
        # Gợi ý: nn.Sequential(nn.Linear(...), nn.GELU(), nn.Linear(...))
        self.net = None  # <- thay
    
    def forward(self, x):
        return self.net(x)

## 4. Positional Encoding
$$\text{PE}(pos, 2i) = \sin(pos / 10000^{2i/d}) \qquad \text{PE}(pos, 2i+1) = \cos(pos / 10000^{2i/d})$$

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()  # (max_len, 1)
        
        # TODO 1: Tính div_term = 10000^(2i/d_model) cho i = 0, 2, 4, ...
        # Gợi ý: dùng torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        #        rồi exp()
        div_term = None  # <- thay
        
        # TODO 2: Gán sin cho index chẵn, cos cho index lẻ
        # Gợi ý: pe[:, 0::2] = torch.sin(position * div_term)
        #        pe[:, 1::2] = torch.cos(position * div_term)
        
        # TODO 3: Thêm batch dimension: (max_len, d_model) -> (1, max_len, d_model)
        # Gợi ý: pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        """
        x: (batch, seq_len, d_model)
        Returns: x + PE (cùng shape)
        """
        # TODO: Cộng positional encoding vào x
        return None  # <- thay

## 5. Encoder Block
```
x → Attn(x) → +x → Norm → FFN → +x → Norm
```

In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        
        # TODO 1: Khởi tạo MultiHeadAttention
        self.attn = None  # <- thay
        
        # TODO 2: Khởi tạo FeedForward
        self.ffn = None  # <- thay
        
        # TODO 3: Khởi tạo 2 LayerNorm
        # Gợi ý: nn.LayerNorm(d_model)
        self.norm1 = None  # <- thay
        self.norm2 = None  # <- thay
        
    def forward(self, x, mask=None):
        """
        x: (batch, seq_len, d_model)
        mask: (batch, 1, 1, seq_len) hoặc None
        Returns: (batch, seq_len, d_model)
        """
        # TODO 1: MultiHeadAttn với residual connection + LayerNorm
        # Công thức: x = norm1(x + attn(x, mask))
        x = None  # <- thay
        
        # TODO 2: FFN với residual connection + LayerNorm
        # Công thức: x = norm2(x + ffn(x))
        x = None  # <- thay
        
        return x

## 6. Transformer Encoder
Stack N encoder blocks + embedding + pos encoding.

In [ ]:
class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, max_len=512):
        super().__init__()
        
        # TODO 1: Token Embedding
        self.embedding = None  # <- nn.Embedding(vocab_size, d_model)
        
        # TODO 2: Positional Encoding
        self.pos_encoding = None  # <- PositionalEncoding(d_model, max_len)
        
        # TODO 3: Stack N EncoderBlock
        # Gợi ý: nn.ModuleList([EncoderBlock(d_model, num_heads, d_ff) for _ in range(num_layers)])
        self.layers = None  # <- thay
        
        # TODO 4: LayerNorm cuối cùng
        self.norm = None  # <- nn.LayerNorm(d_model)
        
    def forward(self, input_ids, attention_mask=None):
        """
        input_ids:      (batch, seq_len)
        attention_mask: (batch, seq_len)  -- 1=real, 0=pad
        Returns:        (batch, seq_len, d_model)
        """
        # TODO 1: Embedding + Positional Encoding
        x = None  # <- embedding + pos_encoding
        
        # TODO 2: Mở rộng attention_mask cho attention
        # Từ (batch, seq_len) -> (batch, 1, 1, seq_len)
        if attention_mask is not None:
            mask = None  # <- thay: attention_mask.unsqueeze(1).unsqueeze(2)
        else:
            mask = None
        
        # TODO 3: Chạy qua từng encoder layer
        for layer in self.layers:
            x = None  # <- thay: layer(x, mask)
        
        # TODO 4: LayerNorm cuối
        x = None  # <- thay: self.norm(x)
        
        return x

## 7. NLI Model
```
input_ids → Encoder → h[CLS] → Linear → entailment/neutral/contradiction
```

In [ ]:
class NLIModel(nn.Module):
    def __init__(self, vocab_size, d_model=256, num_heads=8, d_ff=1024, num_layers=4, num_labels=3):
        super().__init__()
        
        # TODO 1: TransformerEncoder
        self.encoder = None  # <- TransformerEncoder(vocab_size, d_model, num_heads, d_ff, num_layers)
        
        # TODO 2: Classifier head: d_model -> num_labels
        self.classifier = None  # <- nn.Linear(d_model, num_labels)
        
    def forward(self, input_ids, attention_mask=None):
        """
        input_ids:      (batch, seq_len)
        attention_mask: (batch, seq_len)
        Returns:        (batch, num_labels)  -- logits cho 3 nhãn
        """
        # TODO 1: Encode
        # encoder_output: (batch, seq_len, d_model)
        encoder_output = None  # <- self.encoder(input_ids, attention_mask)
        
        # TODO 2: Lấy token đầu tiên ([CLS]) -> (batch, d_model)
        cls_vector = None  # <- encoder_output[:, 0, :]
        
        # TODO 3: Classifier -> (batch, num_labels)
        logits = None  # <- self.classifier(cls_vector)
        
        return logits

In [ ]:
# ============================================================
# 8. TEST: Chạy thử với dữ liệu giả
# ============================================================
# Sau khi implement hết TODOs, chạy cell này để kiểm tra

# Hyperparams nhỏ để test nhanh
VOCAB_SIZE = 1000
BATCH_SIZE = 2
SEQ_LEN = 8

# Tạo dữ liệu giả
dummy_input_ids = torch.randint(0, VOCAB_SIZE, (BATCH_SIZE, SEQ_LEN))
dummy_attn_mask = torch.ones(BATCH_SIZE, SEQ_LEN)
dummy_attn_mask[:, -2:] = 0  # 2 token cuối là PAD
dummy_labels = torch.randint(0, 3, (BATCH_SIZE,))

print("Input shape:", dummy_input_ids.shape)
print("Mask shape:", dummy_attn_mask.shape)
print()

# Khởi tạo model
model = NLIModel(
    vocab_size=VOCAB_SIZE,
    d_model=256,
    num_heads=8,
    d_ff=1024,
    num_layers=4,
    num_labels=3
)

# Forward
logits = model(dummy_input_ids, dummy_attn_mask)
print(f"Logits shape: {logits.shape}")  # Expected: (2, 3)

# Loss
loss_fn = nn.CrossEntropyLoss()
loss = loss_fn(logits, dummy_labels)
print(f"Loss: {loss.item():.4f}")

# Count params
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

# Kiểm tra gradient flow
loss.backward()
grad_count = sum(1 for p in model.parameters() if p.grad is not None)
print(f"Parameters with gradients: {grad_count}")
print()
print("Nếu tất cả chạy không lỗi -> Transformer của bạn đã hoạt động!")